In [ ]:
# NOTE: This data-generation notebook is co-authored and maintained by Claude.
# Per the repo convention the setup / seed notebooks are Claude-owned (user reviews, refines structure,
# and optimizes); the pipeline and transformation code is written by the user.

from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timedelta
import random

In [0]:
# 🏗️ Setup: SwissLogistics Data Platform## Practice Project — Data Generation**Scenario:** SwissLogistics AG is a mid-size logistics company based in Zurich. They operate a fleet of 200+ delivery vehicles across Switzerland, handle ~50K shipments/month, and serve both B2B and B2C customers.**Your role:** You're building their data platform on Databricks. The company has multiple data sources:- **Shipment events** — GPS tracking pings, status updates (picked up, in transit, delivered, failed)- **Customer/partner data** — company profiles that change over time (addresses, tiers, contacts)- **Orders** — order lifecycle with payments, cancellations, returns- **Vehicle telemetry** — sensor data from delivery vehicles (speed, fuel, temperature for cold-chain)**Run this notebook first** to generate all source data. Then work through notebooks 01-08 in order.---### Setup Instructions1. Import all notebooks into a Databricks workspace (Free Edition works)2. Run this notebook to generate source data3. Work through each notebook — they build on each other

In [0]:
# ── Configs ──
ENV = configs.get('env', 'dev')

CATALOG = f"sl_{ENV}"
SCHEMA_INGEST = ENV
SCHEMA_BRONZE = "bronze"  
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
LANDING_VOL = "landing"
LANDING_BASE = f"/Volumes/sl_ingest/{SCHEMA_INGEST}/{LANDING_VOL}"

## Generate Source Data

In [ ]:
# ── ORDERS (50K orders over 12 months) — file-based CDC SNAPSHOT ──
# Orders are delivered as a file-based CDC feed on the landing volume, NOT as a Delta
# table. This cell writes the INITIAL SNAPSHOT: one full row per order, tagged
# _change_type = "SNAPSHOT" (the initial-load marker, analogous to Debezium's "r"/read op).
# The incremental generator later appends INSERT / UPDATE change rows to the same path.
# _change_ts is the sequence column the SCD2 build orders versions by; the snapshot uses
# each order's order_date, so every later change (stamped at batch time) sorts after it.
# Spark's JSON writer drops null fields per row, so later sparse UPDATE rows will carry
# only their key + changed columns, exactly like a real change feed.
products = [
    ("PKG_STD", "Standard Parcel", 12.50), ("PKG_EXP", "Express Parcel", 24.90),
    ("PKG_SAME", "Same-Day Delivery", 39.90), ("FRT_PAL", "Pallet Freight", 89.00),
    ("FRT_FTL", "Full Truck Load", 450.00), ("COLD", "Cold Chain Parcel", 34.90),
    ("DOC", "Document Courier", 8.90), ("INTL", "International Shipment", 65.00),
]
statuses = ["created", "confirmed", "picked_up", "in_transit", "out_for_delivery",
            "delivered", "failed_delivery", "returned"]
payment_methods = ["invoice", "credit_card", "bank_transfer", "paypal"]

order_records = []
for oid in range(1, 50001):
    cid = random.randint(1, 200)
    prod = random.choice(products)
    qty = random.randint(1, 10) if prod[0].startswith("PKG") else random.randint(1, 3)
    order_date = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(6, 22),
                                        minutes=random.randint(0, 59))
    status_idx = random.choices(range(len(statuses)), weights=[5,5,8,15,10,45,7,5])[0]
    status = statuses[status_idx]
    delivery_date = order_date + timedelta(days=random.randint(1, 5)) if status in ("delivered", "returned") else None
    amount = __builtins__.round(prod[2] * qty * random.uniform(0.9, 1.1), 2)
    if random.random() < 0.02:
        amount = None  # ~2% missing amounts (data-quality wrinkle for the silver layer)
    order_records.append((
        f"ORD-{oid:06d}", cid, prod[0], prod[1], qty, amount,
        status, random.choice(payment_methods), order_date, delivery_date,
        random.choice(swiss_cities), random.choice(swiss_cities),
        "SNAPSHOT", order_date,
    ))

order_schema = StructType([
    StructField("order_id", StringType()), StructField("customer_id", IntegerType()),
    StructField("product_code", StringType()), StructField("product_name", StringType()),
    StructField("quantity", IntegerType()), StructField("total_amount", DoubleType(), True),
    StructField("status", StringType()), StructField("payment_method", StringType()),
    StructField("order_date", TimestampType()), StructField("delivery_date", TimestampType(), True),
    StructField("origin_city", StringType()), StructField("destination_city", StringType()),
    StructField("_change_type", StringType()), StructField("_change_ts", TimestampType()),
])

df_orders = spark.createDataFrame(order_records, order_schema)
(df_orders.repartition(10)   # ~5K rows/file
   .write.mode("overwrite")  # initial snapshot seed
   .json(f"{LANDING_BASE}/orders"))
print(f"✓ orders snapshot: {df_orders.count():,} rows -> {LANDING_BASE}/orders (file CDC, _change_type=SNAPSHOT)")


In [0]:
# ── SHIPMENT EVENTS (GPS tracking + status updates, ~500K events) ──
# This is the high-volume streaming-like data

event_types = ["gps_ping", "status_change", "temperature_alert", "delay_alert", "signature_captured"]

shipment_events = []
for i in range(500000):
    oid = f"ORD-{random.randint(1, 50000):06d}"
    vehicle = f"VH-{random.randint(1, 200):03d}"
    evt_type = random.choices(event_types, weights=[60, 20, 5, 10, 5])[0]
    ts = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(0, 23),
                                minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    
    # GPS coordinates (roughly Switzerland: lat 46-48, lon 6-10)
    lat = __builtins__.round(random.uniform(46.0, 47.8), 6) if evt_type == "gps_ping" else None
    lon = __builtins__.round(random.uniform(6.0, 10.5), 6) if evt_type == "gps_ping" else None
    
    # Status for status_change events
    status = random.choice(statuses) if evt_type == "status_change" else None
    
    # Temperature for cold chain alerts
    temp = __builtins__.round(random.gauss(4, 2), 1) if evt_type == "temperature_alert" else None
    
    # Inject duplicates (~3% — common in real event streams)
    shipment_events.append((
        f"EVT-{i+1:07d}", oid, vehicle, evt_type, ts, lat, lon, status, temp,
        random.choice(["kafka", "iot_hub", "api"]),  # source system
    ))
    if random.random() < 0.03:
        shipment_events.append((
            f"EVT-{i+1:07d}", oid, vehicle, evt_type, ts, lat, lon, status, temp,
            random.choice(["kafka", "iot_hub", "api"]),
        ))

event_schema = StructType([
    StructField("event_id", StringType()), StructField("order_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("event_type", StringType()),
    StructField("event_timestamp", TimestampType()),
    StructField("latitude", DoubleType(), True), StructField("longitude", DoubleType(), True),
    StructField("shipment_status", StringType(), True),
    StructField("temperature_celsius", DoubleType(), True),
    StructField("source_system", StringType()),
])

df_events = spark.createDataFrame(shipment_events, event_schema)
(df_events.repartition(100)  # ~5K/file, ~1-2 MB
   .write.mode("overwrite")  # initial seed only
   .json(f"{LANDING_BASE}/shipment_events"))

In [0]:
import uuid
# ── VEHICLE TELEMETRY (sensor readings, ~200K rows, SKEWED by vehicle) ──
# vehicle VH-001 to VH-005 generate 60% of all data (skew for optimization practice)

telemetry = []
for i in range(200000):
    # Intentional skew: 60% of readings from 5 vehicles
    if random.random() < 0.6:
        vehicle = f"VH-{random.randint(1, 5):03d}"
    else:
        vehicle = f"VH-{random.randint(6, 200):03d}"
    
    ts = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(0, 23),
                                minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    telemetry.append((
        str(uuid.uuid4()),                          # reading_id (telematics gateway message id)
        vehicle, ts,
        __builtins__.round(random.uniform(0, 120), 1),       # speed_kmh
        __builtins__.round(random.uniform(0, 100), 1),        # fuel_pct
        __builtins__.round(random.gauss(22, 8), 1),           # engine_temp_c
        __builtins__.round(random.gauss(4, 3), 1) if random.random() < 0.3 else None,  # cargo_temp (only cold chain)
        random.randint(0, 500000),               # odometer_km
        random.choice(["idle", "moving", "loading", "maintenance"]),
    ))

tel_schema = StructType([
    StructField("reading_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("reading_timestamp", TimestampType()),
    StructField("speed_kmh", DoubleType()), StructField("fuel_pct", DoubleType()),
    StructField("engine_temp_c", DoubleType()), StructField("cargo_temp_c", DoubleType(), True),
    StructField("odometer_km", IntegerType()), StructField("vehicle_status", StringType()),
])

df_telemetry = spark.createDataFrame(telemetry, tel_schema)
(df_telemetry.repartition(40)  # ~5K/file, ~1 MB
   .write.mode("overwrite")  # initial seed only
   .json(f"{LANDING_BASE}/telemetry"))

In [ ]:
# ── SUMMARY ──
print("=" * 60)
print("SwissLogistics Data Platform — Source Data Ready")
print("=" * 60)
# Delta source (current-state, CDF on): customers
cust_n = spark.table(f"{SCHEMA_INGEST}.customers").count()
print(f"  {SCHEMA_INGEST}.customers (Delta, CDF): {cust_n:,} rows")
# File-CDC / JSON landing feeds: orders (snapshot), shipment_events, telemetry
for feed in ["orders", "shipment_events", "telemetry"]:
    files = [f for f in dbutils.fs.ls(f"{LANDING_BASE}/{feed}/") if f.name.lower().endswith(".json")]
    rows = spark.read.json(f"{LANDING_BASE}/{feed}/").count()
    print(f"  {LANDING_BASE}/{feed}: {rows:,} rows across {len(files)} files")
print()
print("Data characteristics:")
print("  • customers: current-state source (Postgres-style, CDF on); feeds the SCD2 dimension")
print("  • orders: file-based CDC; SNAPSHOT of 50K orders, ~2% NULL amounts; INSERT/UPDATE drops arrive via the generator")
print("  • shipment_events: ~3% duplicates (dedup practice)")
print("  • vehicle_telemetry: 60% skewed to 5 vehicles (skew handling)")
print()
print("Next: run the bronze ingestion")
